This script downloads the daily and monthly Trinity reservoir data using the scrapper built by WAPA

In [1]:
import requests
import sys
import csv
import json
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import io
import seaborn as sns
import numpy as np
from copy import copy
from scraper import Scraper

In [2]:
#Select the start date
start_string = '1995-10-01'
dsf = "%Y-%m-%d"



In [3]:
# data types that are common to all basins/reservoirs along with the reservoir key
labels_basin = ['snow', 'fnf', 'storage', 'inf', 'otf', 'evap', 'precip', 'gains', 'fci']
stations_basin = ['TRT']


In [4]:
# this initializes the object that will become the input scenario
calfews_data = Scraper(stations_basin, labels_basin, timestep = 'd', start_date = start_string, date_string_format = dsf)


In [5]:
#Checks
calfews_data.data_timeseries

['TRT_snow',
 'TRT_fnf',
 'TRT_storage',
 'TRT_inf',
 'TRT_otf',
 'TRT_evap',
 'TRT_precip',
 'TRT_gains',
 'TRT_fci']

In [6]:
# monthly inputs (extend further back)
start_string_monthly = '1905-10-01'
labels_basin_monthly = ['fnf', 'inf']
calfews_data_m = Scraper(stations_basin, labels_basin_monthly, timestep = 'm', start_date = start_string_monthly, date_string_format = dsf)


In [8]:
#setup input scenario data file with all key/data type pairs (Check)
calfews_data.initialize_dataframe()
calfews_data_m.initialize_dataframe()

           TRT_snow TRT_fnf TRT_storage TRT_inf TRT_otf TRT_evap TRT_precip  \
1995-10-01      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
1995-10-02      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
1995-10-03      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
1995-10-04      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
1995-10-05      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
...             ...     ...         ...     ...     ...      ...        ...   
2025-08-10      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
2025-08-11      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
2025-08-12      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
2025-08-13      NaN     NaN         NaN     NaN     NaN      NaN        NaN   
2025-08-14      NaN     NaN         NaN     NaN     NaN      NaN        NaN   

           TRT_gains TRT_fci  
1995-10-01       NaN

In [11]:
##Monthly Stations Column Lists:
# when CDEC data uses different keys than the main 'basin key'
# we need to associate that key with the main 'basin key' for a specific data type
calfews_data_m.add_station_map(stations_basin, stations_basin, ['inf',])

#Changing Trinity Monthly Inflows to Clair Engle Lake
calfews_data_m.add_station_map(['TRT',], ['CLE',], ['inf',])

In [13]:
# these are the station keys for monthly full-natural flow associated with each key in stations_basin
# TNL - Trinity River at Lewiston has FNF records going back to 1911
stations_monthly_fnf = ['TNL']
calfews_data_m.add_station_map(stations_basin, stations_monthly_fnf, ['fnf',])



In [14]:
#Checks
calfews_data_m.station_use

{'TRT': {'fnf': ['TNL'], 'inf': ['CLE']}}

In [15]:
# these are the station keys for daily downstream incremental flows associated with each key in stations_basin
#None added for Trinity in second position
station_names = ['none']
station_types = ['gains',]
calfews_data.add_station_map(stations_basin, station_names, station_types)

In [16]:
#Checks
calfews_data.station_use

{'TRT': {'snow': ['TRT'],
  'fnf': ['TRT'],
  'storage': ['TRT'],
  'inf': ['TRT'],
  'otf': ['TRT'],
  'evap': ['TRT'],
  'precip': ['TRT'],
  'gains': ['none'],
  'fci': ['TRT']}}

In [17]:
# different station keys for Trinity
calfews_data.add_station_map(['TRT',], ['TNL'], ['fnf',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['storage',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['inf',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['otf',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['evap',])
calfews_data.add_station_map(['TRT',], ['CLE'], ['precip',])
calfews_data.add_station_map(['TRT',], ['none'], ['fci',]) #--- This is hard coded in the model where the fcr is computed

In [18]:
#Checks
calfews_data.station_use

{'TRT': {'snow': ['TRT'],
  'fnf': ['TNL'],
  'storage': ['CLE'],
  'inf': ['CLE'],
  'otf': ['CLE'],
  'evap': ['CLE'],
  'precip': ['CLE'],
  'gains': ['none'],
  'fci': ['none']}}

In [19]:
# snow station mapping (Trinity Values are added)
basin_names = ['TRT']
station_names = [['MUM', 'BNK', 'RRM']]
calfews_data.add_station_map(basin_names, station_names, ['snow',])

In [20]:
##basin-level data
sensor_basin = [82, 8, 15, 76, 23, 74, 45, 41]
labels_basin = ['snow', 'fnf', 'storage', 'inf', 'otf', 'evap', 'precip', 'gains']
url_parts_d = ['https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet?Stations=', '&SensorNums=', '&dur_code=D&Start=', '&End=']
url_parts_m = ['https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet?Stations=', '&SensorNums=', '&dur_code=M&Start=', '&End=']
url_parts_h = ['https://cdec.water.ca.gov/dynamicapp/req/CSVDataServlet?Stations=', '&SensorNums=', '&dur_code=H&Start=', '&End=']


In [21]:
hourly_dict = {}
for x in stations_basin:
  hourly_dict[x] = []

In [22]:
#Checks
calfews_data.station_use

{'TRT': {'snow': ['MUM', 'BNK', 'RRM'],
  'fnf': ['TNL'],
  'storage': ['CLE'],
  'inf': ['CLE'],
  'otf': ['CLE'],
  'evap': ['CLE'],
  'precip': ['CLE'],
  'gains': ['none'],
  'fci': ['none']}}

In [23]:
# link and read all data
calfews_data_m.link_api(stations_basin, [76,], ['inf',], url_parts_m, url_parts_h, hourly_dict)
calfews_data_m.link_api(stations_basin, [65,], ['fnf',], url_parts_m, url_parts_h, hourly_dict)
calfews_data.link_api(stations_basin, sensor_basin, labels_basin, url_parts_d, url_parts_h, hourly_dict)
calfews_data_m.find_ratios(stations_basin, 'inf')
calfews_data.fill_missing(stations_basin, calfews_data_m.station_coefs, calfews_data_m.real_time_data)
calfews_data.adjust_fnf_monthly(calfews_data_m.real_time_data, stations_basin)


Start Link: TRT inf (CDEC Station: CLE 76)
Start Link: TRT fnf (CDEC Station: TNL 65)
Start Link: TRT snow (CDEC Station: MUM 82)
Start Link: TRT snow (CDEC Station: BNK 82)
Start Link: TRT snow (CDEC Station: RRM 82)
Start Link: TRT fnf (CDEC Station: TNL 8)
Start Link: TRT storage (CDEC Station: CLE 15)
Start Link: TRT inf (CDEC Station: CLE 76)
Start Link: TRT otf (CDEC Station: CLE 23)
Start Link: TRT evap (CDEC Station: CLE 74)
Start Link: TRT precip (CDEC Station: CLE 45)


In [24]:
calfews_data



In [27]:
calfews_data.real_time_data
#calfews_data_m.real_time_data
#calfews_data_m.real_time_data.to_csv('cord-sim_realtime_monthly.csv')

,TRT_snow,TRT_fnf,TRT_storage,TRT_inf,TRT_otf,TRT_evap,TRT_precip,TRT_gains,TRT_fci
1995-10-01,0.00,73.354839,1869700.0,0.000000,0.0,41.0,0.0,0.0,0.0
1995-10-02,0.00,73.354839,1866506.0,26.250000,1642.0,81.0,0.0,0.0,0.0
1995-10-03,0.00,73.354839,1863050.0,26.250000,1839.0,53.0,0.0,0.0,0.0
1995-10-04,0.00,73.354839,1855603.0,8.166667,3833.0,90.0,0.0,0.0,0.0
1995-10-05,0.00,73.354839,1849828.0,8.166667,2486.0,53.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
2025-08-10,124.81,0.000000,2039553.0,396.000000,2262.0,129.0,0.0,0.0,0.0
2025-08-11,124.81,0.000000,2036767.0,579.000000,1850.0,134.0,0.0,0.0,0.0
2025-08-12,124.81,0.000000,2032515.0,266.000000,2281.0,129.0,0.0,0.0,0.0
2025-08-13,124.81,0.000000,2028110.0,266.000000,2281.0,0.0,0.0,0.0,0.0
